# CV Matching — notebook Colab exécutable de haut en bas
 Livrable expérimental UIR : tout le pipeline sans Docker. Runtime recommandé : GPU (accélère SBERT).
 Les résultats affichés sont **mesurés sur place** : rien n'est pré-rempli.

In [ ]:
import os, sys, zipfile
from pathlib import Path
if not Path("pyproject.toml").exists():
    try:  # Colab : téléverser le zip du projet
        from google.colab import files
        up = files.upload()
        name = next(iter(up))
        zipfile.ZipFile(name).extractall(".")
        os.chdir(next(p for p in Path(".").iterdir() if (p / "pyproject.toml").exists()))
    except ImportError:
        raise SystemExit("Ouvrez ce notebook depuis la racine du projet.")
print(os.getcwd())

In [ ]:
%pip install -q -e . datasets matplotlib

## 1. Données : téléchargement Hugging Face + nettoyage (M1.1 / M1.2)

In [ ]:
!python scripts/prepare_data.py --max-rows 8000

## 2. Embeddings Sentence-BERT des offres (calculés une fois)

In [ ]:
!python scripts/build_artifacts.py

## 3. CV de test fictifs et pool d'annotation

In [ ]:
!python scripts/make_demo_cvs.py
!python scripts/make_eval_pool.py

## 4. Ground truth
 Annotez `data/eval/pool_to_annotate.csv` (colonnes `annotator_1` et `annotator_2`, 1 = pertinente, 0 = non), puis
 lancez `python scripts/finalize_labels.py`. **Sans annotation**, la cellule suivante bascule sur des labels
 *proxy* (mots-clés) : utile pour vérifier que la chaîne tourne, **pas pour rapporter des résultats**.

In [ ]:
from pathlib import Path
use_proxy = not Path("data/eval/labels.csv").exists()
print("⚠ labels PROXY (test technique)" if use_proxy else "✔ labels annotés")

In [ ]:
!python scripts/run_ablation.py {"--proxy" if use_proxy else ""}

## 5. Démonstration : Top 3 + explications

In [ ]:
from cvmatch.config import get_settings
from cvmatch.service import build_service
s = get_settings()                       # RANKER / EXPLAINER via .env ou variables d'environnement
svc = build_service(s)
cv = Path("data/eval/cvs/cv_01.txt").read_text(encoding="utf-8")
res = svc.match(cv, top_k=3, explain=True, explainer="rules")
for r in res.recommendations:
    print(f"{r.similarity_score:.3f}  {r.job_title}  [{r.company}]")
    print("   ", r.explanation.explanation)

## 6. Gemini (optionnel)
 Colab : menu 🔑 *Secrets* → `GEMINI_API_KEY`. Ne jamais écrire la clé dans le notebook.

In [ ]:
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception as e:
    print("Pas de clé Colab :", e)
s = get_settings()
svc = build_service(s)
res = svc.match(cv, explainer="gemini")
print(res.warnings or "Gemini OK")
print(res.recommendations[0].explanation.model_dump())

In [ ]:
!python scripts/run_llm_ablation.py --ranker tfidf --n 20 --no-cache